# Run a point-in-time prompt through the front

The front is one call, `ask`. It takes a prompt, a decision definition and the
id of a catalogue entry, and returns the answer together with the entry, mode
and status it ran under and what limited it. This notebook lists the catalogue,
selects an entry, runs the worked example and reads the result.

It runs offline by default: nothing is sent, and every draw gets the same
invented fixed reply, written by hand. The run then shows the wiring and how to
read a result. It says nothing about a model's quality.

Select the Python environment of the development checkout as the kernel. The
companion script is `examples/pit_front.py`.

In [ ]:
FIXTURE = True  # False reaches the model through the entry's provider profile:
                # that route must be up and the environment its profile names must be set.
ENTRY = "llamacpp-gemma3-4b"  # The entry that offers both modes.
MODE = None  # None lets the front choose; or "logprobs" or "logits".

## The catalogue

The catalogue lists the models Recall Guard has records for, each on one serving
route. An entry has two evidence modes. The logprobs mode reads the token
log probabilities the route returns with a reply. The logits mode reads the full
next-token distribution from a host that serves it. A route offers a mode or it
does not.

In [ ]:
from recall_guard import WORKED_EXAMPLE_REPLY, FixedReplies, Mode, Status, ask, load_catalogue, worked_example


def words(status):
    return status.value.replace("_", " ")


catalogue = load_catalogue()
print(f"{len(catalogue.entries)} entries; {len(catalogue.filter(status=Status.QUALIFIED))} with a qualified mode")
print()
print(f"{'entry':<29}{'logprobs':<15}{'logits':<15}model on route")
for entry in catalogue.entries:
    cells = [words(catalogue.qualification(entry.entry_id, mode).status) if entry.modes[mode].available
             else "not offered" for mode in Mode]
    print(f"{entry.entry_id:<29}{cells[0]:<15}{cells[1]:<15}{entry.model} on {entry.route}")

## What the statuses mean

Each entry and mode has one status, decided by a fixed rule from the recorded
calibration.

- **qualified**: the recorded calibration passed every check. The front can
  correct the answers of this entry and mode.
- **not qualified**: a recorded check failed. The reason names it.
- **not tested**: a check has no recorded value, so the rule cannot decide.

A status never refuses an answer. An entry that is not qualified or not tested
still answers, uncorrected and marked with its status.

In [ ]:
selected = catalogue.entry(ENTRY)
print(f"{selected.entry_id}: {selected.model} on {selected.route}")
for mode in Mode:
    record = selected.modes[mode]
    qualification = catalogue.qualification(ENTRY, mode)
    print(f"  {mode.value}: " + (f"{words(qualification.status)} ({qualification.reason})" if record.available
                                 else f"not offered ({record.unavailable_reason})"))

## A fenced point-in-time prompt

A point-in-time answer should come from the state shown in the prompt and not
from what the model remembers about a period. A fenced prompt removes the cues
that identify the period: it shows the macro state as z-scores against a rolling
window and the assets by letter and category, with no date, no ticker and no raw
level. It asks for the regime as loadings on five axes and for no forecast.

`worked_example()` returns such a prompt with its decision definition, which
says how a reply is read into an answer. The values in the prompt are invented.
The front accepts any prompt with a decision definition; this one ships with the
package.

In [ ]:
prompt, definition = worked_example()
print(prompt)

## Run it through the front

With `client=None` the front builds the client from the entry's provider
profile. Offline, the fixed client replays one reply for each of the 32 draws.
The front then reduces the draws to one answer with the decision definition.

In [ ]:
client = FixedReplies([WORKED_EXAMPLE_REPLY], cycle=True) if FIXTURE else None
result = ask(prompt, definition, entry=ENTRY, mode=None if MODE is None else Mode(MODE), client=client,
             catalogue=catalogue)
if FIXTURE:
    print(f"Nothing was sent. The fixed reply: {WORKED_EXAMPLE_REPLY}")

## Read the result

Read it top to bottom: the entry and the mode the answer ran under, with the
rule that chose the mode; the status and its reason; the answer; the recall
estimate; the corrected answer; the limits. When the front cannot produce an
answer it returns no partial one. It names the step that failed and what to
change.

In [ ]:
def loadings(decision):
    return [f"  {name}: {'none' if value is None else format(value, '+.2f')}" for name, value in decision.items()]


print(f"entry: {result.entry_id}")
if result.mode is not None:
    print(f"mode: {result.mode.value} ({result.mode_choice})")
if result.status is not None:
    print(f"status: {words(result.status)}")
    print(f"reason: {result.reason}")
if result.failure is not None:
    print("no answer")
    print(f"failed at step: {result.failure.step}")
    print(f"detail: {result.failure.detail}")
    print(f"what to change: {result.failure.remedy}")
else:
    print("answer (loading on each axis, from -1 to +1):", *loadings(result.answer), sep="\n")
    print("recall estimate: " + ("none" if result.p_memorized is None else f"{result.p_memorized:.3f}"))
    if result.corrected is not None:
        print("corrected answer:", *loadings(result.corrected), sep="\n")
    elif result.status is Status.QUALIFIED:
        print("corrected answer: none, because the run did not meet what a correction needs; "
              "see the reason and the limits")
    else:
        print(f"corrected answer: none, because the status is {words(result.status)} "
              "and only a qualified entry and mode is corrected")
    if result.limits:
        print("limits:", *(f"  - {limit}" for limit in result.limits), sep="\n")
    else:
        print("limits: none")

## What the limits say

Each limit is something the answer lacks, in a fixed phrase. An empty list means
nothing limited the answer. Here the entry has no qualified mode and no
calibrator was supplied, so the answer is the model's reply to a fenced prompt
and nothing more: there is no recall estimate, so nothing says how much of the
answer is recall, and there is no correction.

No calibrator is fitted in this notebook, because fitting needs a calibration
set and a model. A calibrator fitted for the entry's model and mode and passed
to `ask` adds the recall estimate, and for a qualified entry and mode the
corrected answer. `docs/pit-architecture.md` explains the calibration.